# BT5153 Autonomous Uplift Pipeline Demo

This notebook is a presentation wrapper around the tested `src/` code. It uses the offline stub LLM by default, so the whole loop can run without network access or API keys. To experiment with hosted LLM planning and evaluation, copy `.env.example` to `.env` and set the provider/key/model values there.


```mermaid
flowchart TD
    data[RetailHero CSVs] --> contract[UpliftProjectContract]
    contract --> validation[Validation and balance diagnostics]
    contract --> features[Cached feature artifacts]
    features --> planner[Planning phase]
    planner --> loop[Deterministic trial loop]
    loop --> ledger[JSONL ledger]
    ledger --> evaluation[Judge / XAI / policy]
    evaluation --> retry[RetryControllerAgent]
    retry --> planner
    ledger --> report[Final report]
    features --> submission[client_id,uplift submission]
```


## 1. Runtime Settings

The notebook reads `.env` by default. Use `RETAILHERO_DATA_DIR=retailhero-uplift/data` for the real run; fixture data should only be used for CI and quick smoke checks. Generated artifacts are written under `artifacts/uplift/autonomous_notebook_*`.


In [ ]:
from __future__ import annotations

import os
import pickle
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'src').exists() and (candidate / 'tests').exists():
            return candidate
    raise RuntimeError('Could not find repo root containing src/ and tests/.')


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def load_local_env(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('\"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_local_env(ROOT / ".env")

USE_TINY_FIXTURE = os.getenv('BT5153_USE_TINY_FIXTURE', 'true').lower() == 'true'
PROVIDER = os.getenv('LLM_PROVIDER', 'stub').strip().lower()
BASE_MODEL = os.getenv('LLM_MODEL') or None
PLANNING_MODEL = os.getenv('LLM_PLANNING_MODEL') or BASE_MODEL
EVALUATION_MODEL = os.getenv('LLM_EVALUATION_MODEL') or BASE_MODEL
MAX_ITERATIONS = int(os.getenv('BT5153_MAX_ITERATIONS', '3'))
RUN_BENCHMARK = os.getenv('BT5153_RUN_BENCHMARK', 'true').lower() == 'true'

API_KEY_ENV_BY_PROVIDER = {
    'openai': 'OPENAI_API_KEY',
    'gemini': 'GEMINI_API_KEY',
    'claude': 'ANTHROPIC_API_KEY',
}
API_KEY_ENV = API_KEY_ENV_BY_PROVIDER.get(PROVIDER)
API_KEY = os.getenv(API_KEY_ENV) if API_KEY_ENV else None
if API_KEY_ENV and not API_KEY:
    raise RuntimeError(f'Set {API_KEY_ENV} in .env before using provider={PROVIDER!r}.')

DATA_DIR = Path(os.getenv('RETAILHERO_DATA_DIR', 'tests/fixtures/uplift'))
if not DATA_DIR.is_absolute():
    DATA_DIR = ROOT / DATA_DIR

run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = ROOT / 'artifacts' / 'uplift' / f'autonomous_notebook_{run_stamp}'

settings = pd.DataFrame(
    [
        {'setting': 'repo_root', 'value': str(ROOT)},
        {'setting': 'data_dir', 'value': str(DATA_DIR)},
        {'setting': 'output_dir', 'value': str(OUTPUT_DIR)},
        {'setting': 'provider', 'value': PROVIDER},
        {'setting': 'base_model', 'value': BASE_MODEL or '(provider default)'},
        {'setting': 'planning_model', 'value': PLANNING_MODEL or '(provider default)'},
        {'setting': 'evaluation_model', 'value': EVALUATION_MODEL or '(provider default)'},
        {'setting': 'max_iterations', 'value': MAX_ITERATIONS},
        {'setting': 'run_benchmark', 'value': RUN_BENCHMARK},
    ]
)
settings


## 2. Build the Source-of-Truth Contract

The contract owns table paths, target/treatment semantics, split policy, evaluation policy, and submission semantics. Hosted LLMs can suggest plans, but they do not change this contract.


In [ ]:
from src.models.uplift import (
    UpliftFeatureRecipeSpec,
    UpliftProjectContract,
    UpliftSplitContract,
    UpliftTableSchema,
    UpliftTrialSpec,
)
from src.uplift.features import build_feature_table
from src.uplift.hypotheses import UpliftHypothesisStore
from src.uplift.ledger import UpliftLedger
from src.uplift.llm_client import make_chat_llm
from src.uplift.orchestrator import AutoLiftOrchestrator, RetryControllerAgent
from src.uplift.planning_agents import ExperimentPlanningPhase
from src.uplift.reporting import generate_submission_artifact, validate_submission_artifact
from src.uplift.validation import compute_treatment_control_balance, validate_uplift_dataset


def build_contract(data_dir: Path) -> UpliftProjectContract:
    split_contract = (
        UpliftSplitContract(
            train_fraction=0.5,
            val_fraction=0.5,
            test_fraction=0.0,
            min_rows_per_partition=1,
            random_seed=7,
        )
        if USE_TINY_FIXTURE
        else UpliftSplitContract()
    )
    return UpliftProjectContract(
        task_name='retailhero-uplift',
        description='BT5153 autonomous uplift experiment demo.',
        table_schema=UpliftTableSchema(
            clients_table=str(data_dir / 'clients.csv'),
            purchases_table=str(data_dir / 'purchases.csv'),
            products_table=str(data_dir / 'products.csv'),
            train_table=str(data_dir / 'uplift_train.csv'),
            scoring_table=str(data_dir / 'uplift_test.csv'),
        ),
        split_contract=split_contract,
        feature_sources=['clients', 'purchases', 'products'],
    )


contract = build_contract(DATA_DIR)
contract.model_dump()


## 3. Validate the Dataset

This confirms the labeled/scoring table contract before any model or agent run starts.


In [ ]:
validation_report = validate_uplift_dataset(contract)
train_df = pd.read_csv(contract.table_schema.train_table)
balance_report = compute_treatment_control_balance(
    train_df,
    entity_key=contract.entity_key,
    treatment_col=contract.treatment_column,
    target_col=contract.target_column,
)

pd.DataFrame(
    [
        {'check': 'dataset_valid', 'value': validation_report.valid},
        {'check': 'errors', 'value': validation_report.errors},
        {'check': 'table_rows', 'value': validation_report.table_rows},
        {'check': 'treatment_counts', 'value': validation_report.treatment_counts},
        {'check': 'target_counts', 'value': validation_report.target_counts},
        {'check': 'balance_warnings', 'value': balance_report.warnings},
    ]
)


## 4. Build Cached Feature Artifacts

The notebook uses one approved RFM-style recipe for both labeled training customers and unlabeled scoring customers. The scoring artifact is kept target/treatment-free for submission.


In [ ]:
recipe = UpliftFeatureRecipeSpec(
    source_tables=['clients', 'purchases'],
    feature_groups=['demographic', 'rfm', 'basket', 'points'],
    windows_days=[30, 60, 90],
    builder_version='v1',
)

feature_dir = OUTPUT_DIR / 'features'
train_artifact = build_feature_table(
    contract,
    recipe=recipe,
    output_dir=feature_dir,
    cohort='train',
    chunksize=10,
    progress_logger=print,
)
scoring_artifact = build_feature_table(
    contract,
    recipe=recipe,
    output_dir=feature_dir,
    cohort='scoring',
    chunksize=10,
    progress_logger=print,
)

pd.DataFrame(
    [
        train_artifact.model_dump(),
        scoring_artifact.model_dump(),
    ]
)[['cohort', 'row_count', 'feature_recipe_id', 'feature_artifact_id', 'artifact_path']]


## 5. Run the Autonomous Loop

`AutoLiftOrchestrator` wires together PR2 planning, deterministic trial execution, Judge/XAI/Policy evaluation, `RetryControllerAgent`, and final report generation. Planning and evaluation can use different model IDs from `.env`, which lets OpenAI runs use a stronger reasoning model for analytical planning while keeping evaluation cheaper.


In [ ]:
planner_llm = make_chat_llm(PROVIDER, PLANNING_MODEL, API_KEY)
evaluation_llm = make_chat_llm(PROVIDER, EVALUATION_MODEL, API_KEY)
run_dir = OUTPUT_DIR / 'runs'
ledger = UpliftLedger(run_dir / 'uplift_ledger.jsonl')
hypothesis_store = UpliftHypothesisStore(OUTPUT_DIR / 'hypotheses.jsonl')
planner = ExperimentPlanningPhase(ledger, hypothesis_store, planner_llm)

orchestrator = AutoLiftOrchestrator(
    contract=contract,
    planner=planner,
    feature_artifacts_by_name={'rfm_baseline': train_artifact},
    output_dir=run_dir,
    llm=evaluation_llm,
    run_benchmark=RUN_BENCHMARK,
)

result = orchestrator.run(max_iterations=MAX_ITERATIONS)
retry_snapshot = RetryControllerAgent(ledger).run()

pd.DataFrame(
    [
        {'field': 'agent_trials', 'value': len(result.trial_records)},
        {'field': 'benchmark_run_id', 'value': result.benchmark_record.run_id if result.benchmark_record else None},
        {'field': 'retry_should_continue', 'value': result.retry_decision.should_continue},
        {'field': 'retry_reason', 'value': result.retry_decision.reason},
        {'field': 'retry_next_action', 'value': result.retry_decision.suggested_next_action},
        {'field': 'fresh_retry_snapshot', 'value': retry_snapshot.reason},
        {'field': 'report_path', 'value': result.report_path},
    ]
)


## 6. Inspect the Ledger

The ledger is the evidence spine for reporting, retry decisions, and future planning context.


In [ ]:
records = ledger.load()
ledger_df = pd.DataFrame(
    [
        {
            'run_id': record.run_id,
            'hypothesis_id': record.hypothesis_id,
            'template': record.template_name,
            'learner': record.uplift_learner_family,
            'base_estimator': record.base_estimator,
            'status': record.status,
            'qini_auc': record.qini_auc,
            'uplift_auc': record.uplift_auc,
            'policy_gain': record.policy_gain,
        }
        for record in records
    ]
)
ledger_df.sort_values(['status', 'qini_auc'], ascending=[True, False])


## 7. Review Evaluation Outputs

Each successful autonomous trial receives bounded judge output, XAI support, and policy simulation output. The deterministic metrics remain authoritative.


In [ ]:
evaluation_rows = []
for index, evaluation in enumerate(result.evaluation_results, start=1):
    judge = evaluation.get('judge', {})
    xai = evaluation.get('xai', {})
    policy = evaluation.get('policy', {})
    evaluation_rows.append(
        {
            'trial_index': index,
            'judge_verdict': judge.get('verdict'),
            'qini_auc': judge.get('computed_metrics', {}).get('qini_auc'),
            'xai_method': xai.get('method') or xai.get('reason'),
            'leakage_flag': xai.get('leakage_auto_flag'),
            'recommended_threshold_pct': policy.get('recommended_threshold'),
            'policy_summary_keys': sorted(policy.get('policy_data', {}).keys()),
        }
    )

pd.DataFrame(evaluation_rows)


## 8. Preview the Final Report

The report is generated from ledger records and evaluation outputs, not from notebook-only state.


In [ ]:
report_text = Path(result.report_path).read_text(encoding='utf-8') if result.report_path else ''
print(report_text[:4000])


## 9. Generate a Submission Preview

The submission preview scores `uplift_test.csv` customers with the best successful autonomous trial model. It validates exact `client_id,uplift` schema and scoring-customer coverage.


In [ ]:
successful_agent_records = [record for record in result.trial_records if record.status == 'success']
if not successful_agent_records:
    raise RuntimeError('No successful autonomous trial is available for submission scoring.')

champion = max(
    successful_agent_records,
    key=lambda record: record.qini_auc if record.qini_auc is not None else float('-inf'),
)
model_path = Path(champion.artifact_paths['model'])
with model_path.open('rb') as handle:
    champion_model = pickle.load(handle)

champion_trial = UpliftTrialSpec(
    spec_id=champion.hypothesis_id,
    hypothesis_id=champion.hypothesis_id,
    template_name=champion.template_name,
    learner_family=champion.uplift_learner_family,
    base_estimator=champion.base_estimator,
    feature_recipe_id=champion.feature_recipe_id,
    params={},
    split_seed=champion.split_seed,
)

submission = generate_submission_artifact(
    contract,
    model=champion_model,
    scoring_feature_artifact=scoring_artifact,
    champion_trial=champion_trial,
    output_path=OUTPUT_DIR / 'uplift_submission.csv',
)
validate_submission_artifact(contract, submission)

pd.read_csv(submission.artifact_path).head()


## 10. Artifact Index

Use these paths for screenshots, report excerpts, or post-demo inspection.


In [ ]:
pd.DataFrame(
    [
        {'artifact': 'train_features', 'path': train_artifact.artifact_path},
        {'artifact': 'scoring_features', 'path': scoring_artifact.artifact_path},
        {'artifact': 'ledger', 'path': str(ledger.path)},
        {'artifact': 'report', 'path': result.report_path},
        {'artifact': 'submission', 'path': submission.artifact_path},
        {'artifact': 'hypotheses', 'path': str(hypothesis_store.path)},
    ]
)
